# Earth System Grid Federation Data Access

The simplest way to search datasets programmatically within the Earth System Grid Federation (ESGF) is to use [`intake-esgf`](https://intake-esgf.readthedocs.io/). It works like `intake-esm`, but instead of loading a static catalog, `intake-esgf` populates its catalog by querying ESGF index nodes. There is also a new graphical search interface called [Metagrid](https://aims2.llnl.gov/search), but it does not yet federate data from all ESGF nodes.

ESGF provides two different mechanisms to retrieve data, the legacy `Solr` index and the new `Globus` index. ESGF is decommissioning the old indexing service based on SOLR, so methods relying on it might break. `intake-esgf` works with both, so should allow for a graceful transition.

What `intake-esgf` does is send search queries to a list of index nodes and aggregates the results. You can configure the list of index nodes it queries, especially since some of the nodes are very slow to respond. For this notebook, we'll query only Globus nodes to speed things up, but to ensure you find all the available data, call `intake_esgf.conf.set(all_indices=True)` to search all nodes.

In [ ]:
# NBVAL_IGNORE_OUTPUT

import intake_esgf

# Use only Globus index nodes to speed things up
intake_esgf.conf.set({"solr_indices": {}})

# Silence warnings from STAC publishing error
intake_esgf.conf.set(
    {"indices": {"discovery.east.esgf.io": False, "discovery.west.esgf.io": False}}
)

# Show the configuration
print(intake_esgf.conf)

# Initialize an empty ESGF catalog
cat = intake_esgf.ESGFCatalog()

# Launch a search query.
# Here we're looking for any variable related to humidity within the CMIP6 SSP2-4.5 experiment.
# Results will be stored in a dictionary with keys defined by the `facets` argument.
cat.search(
    project="CMIP6",
    variable_id=["hurs"],
    table_id="Amon",
    experiment_id=["ssp126", "ssp245", "ssp370", "ssp585"],
)

additional_df_cols: []
break_on_error: true
confirm_download: false
default_open_kwargs:
  chunks: auto
  compat: override
  coords: minimal
  data_vars: minimal
default_project: CMIP7
download_db: ~/.config/intake-esgf/download.db
esg_dataroot:
- /p/css03/esgf_publish
- /eagle/projects/ESGF2/esg_dataroot
- /global/cfs/projectdirs/m3522/cmip6/
- /glade/campaign/collections/cmip.mirror
globus_indices:
  ESGF2-US-1.5-Catalog: true
local_cache:
- ~/.esgf/
logfile: ~/.config/intake-esgf/esgf.log
num_threads: 6
print_log_on_error: false
requests_cache:
  cache_name: intake-esgf/requests-cache.sqlite
  expire_after: 3600
  use_cache_dir: true
slow_download_threshold: 0.5
solr_indices: {}
stac_indices:
  discovery.east.esgf.io: true
  discovery.west.esgf.io: true



/opt/conda/envs/birdy/lib/python3.12/site-packages/intake_esgf/catalog.py:936: UserWarning: You are searching for project='CMIP6' with at least one STAC-based index enabled. While republishing CMIP6 data to STAC is underway and may work well, we have found a publishing error that affects many records. You may see some unexpected `member_id` fields. Temporarily, we recommend disabling these indices for work with CMIP6:
intake_esgf.conf.set(indices={'discovery.east.esgf.io':False,'discovery.west.esgf.io': False})
  warnings.warn(


Output()

Summary information for 1705 results:
mip_era                                                     [CMIP6]
activity_id                                           [ScenarioMIP]
institution_id    [AWI, CAS, CCCR-IITM, CCCma, CMCC, CNRM-CERFAC...
source_id         [AWI-CM-1-1-MR, CAS-ESM2-0, FGOALS-f3-L, FGOAL...
experiment_id                      [ssp126, ssp245, ssp370, ssp585]
member_id         [r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p1f1, r5i1p...
table_id                                                     [Amon]
variable_id                                                  [hurs]
grid_label                                            [gn, gr, gr1]
dtype: object

You can get a sense of what datasets are available with the `model_group` method, which counts the number of unique combinations of `source_id, member_id` and `grid_label`.

Other useful methods are `remove_ensembles`, which picks only one `member_id` (the smallest 4 integer values for the *ripf* code, and `remove_incomplete`, which filters model groups according to criteria you can define. See the [docs](https://intake-esgf.readthedocs.io/en/latest/modelgroups.html) for details.

In [2]:
# NBVAL_IGNORE_OUTPUT

# Keep only one member per model, experiment and grid.
cat.remove_ensembles()

# Remove models groups that don't have the four SSPs.
cat.remove_incomplete(lambda df: len(df) == 4)

cat.model_groups()

source_id         member_id  grid_label
ACCESS-CM2        r1i1p1f1   gn            4
ACCESS-ESM1-5     r1i1p1f1   gn            4
AWI-CM-1-1-MR     r1i1p1f1   gn            4
CanESM5           r1i1p1f1   gn            4
CanESM5-1         r1i1p1f1   gn            4
CanESM5-CanOE     r1i1p2f1   gn            4
CAS-ESM2-0        r1i1p1f1   gn            4
CESM2-WACCM       r1i1p1f1   gn            4
CMCC-CM2-SR5      r1i1p1f1   gn            4
CMCC-ESM2         r1i1p1f1   gn            4
CNRM-CM6-1        r1i1p1f2   gr            4
CNRM-CM6-1-HR     r1i1p1f2   gr            4
CNRM-ESM2-1       r1i1p1f2   gr            4
EC-Earth3         r1i1p1f1   gr            4
EC-Earth3-Veg     r1i1p1f1   gr            4
EC-Earth3-Veg-LR  r1i1p1f1   gr            4
FGOALS-f3-L       r1i1p1f1   gr            4
FGOALS-g3         r1i1p1f1   gn            4
GFDL-ESM4         r1i1p1f1   gr1           4
GISS-E2-1-G       r1i1p1f2   gn            4
GISS-E2-1-H       r1i1p1f2   gn            4
GISS-E2-2-G    

Now we'll try to access some data. For small queries, a good approach is to use streaming, rather than downloading the whole thing. Here we'll just ask for simulations from CanESM5, and try to stream some data. Getting the file information can take some time.

In [3]:
# NBVAL_IGNORE_OUTPUT

# Let's focus the search on one single model to speed up the rest of the notebook
cat.search(
    project="CMIP6",
    source_id="CanESM5",
    variable_id=["hurs"],
    table_id="Amon",
    experiment_id=["ssp126", "ssp245", "ssp370", "ssp585"],
)
cat.remove_ensembles()

# The `prefer_streaming` argument specifies that we'd rather not download entire files.
# When True, the `add_measures` argument triggers search for variables that are referenced
# in the `cell_measure` attribute, such as `areacella` or `areacello`.
dsd = cat.to_dataset_dict(prefer_streaming=True, add_measures=False)

Output()

/opt/conda/envs/birdy/lib/python3.12/site-packages/intake_esgf/catalog.py:936: UserWarning: You are searching for project='CMIP6' with at least one STAC-based index enabled. While republishing CMIP6 data to STAC is underway and may work well, we have found a publishing error that affects many records. You may see some unexpected `member_id` fields. Temporarily, we recommend disabling these indices for work with CMIP6:
intake_esgf.conf.set(indices={'discovery.east.esgf.io':False,'discovery.west.esgf.io': False})
  warnings.warn(


Output()

/opt/conda/envs/birdy/lib/python3.12/site-packages/intake_esgf/catalog.py:747: UserWarning: xarray threw an exception opening these files: [PosixPath('/home/jenkins/.esgf/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp585/r1i1p1f1/Amon/hurs/gn/v20190429/hurs_Amon_CanESM5_ssp585_r1i1p1f1_gn_201501-210012.nc'), PosixPath('/home/jenkins/.esgf/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp585/r1i1p1f1/Amon/hurs/gn/v20190429/hurs_Amon_CanESM5_ssp585_r1i1p1f1_gn_210101-218012.nc'), PosixPath('/home/jenkins/.esgf/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp585/r1i1p1f1/Amon/hurs/gn/v20190429/hurs_Amon_CanESM5_ssp585_r1i1p1f1_gn_218101-230012.nc')]
  warnings.warn(f"xarray threw an exception opening these files: {files}")
/opt/conda/envs/birdy/lib/python3.12/site-packages/intake_esgf/catalog.py:747: UserWarning: xarray threw an exception opening these files: [PosixPath('/home/jenkins/.esgf/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp370/r1i1p1f1/Amon/hurs/gn/v20190429/hurs_Amon_CanESM5_ssp370_r1i1p1f1_gn_201501-210012.nc')]
  warning

Can not use auto rechunking with object dtype. We are unable to estimate the size in bytes of object data
Can not use auto rechunking with object dtype. We are unable to estimate the size in bytes of object data
Can not use auto rechunking with object dtype. We are unable to estimate the size in bytes of object data
Can not use auto rechunking with object dtype. We are unable to estimate the size in bytes of object data


DatasetInitError: xarray threw an exception while loading these keys: ['CMIP6.ScenarioMIP.CCCma.CanESM5.ssp585.r1i1p1f1.Amon.hurs.gn', 'CMIP6.ScenarioMIP.CCCma.CanESM5.ssp370.r1i1p1f1.Amon.hurs.gn', 'CMIP6.ScenarioMIP.CCCma.CanESM5.ssp245.r1i1p1f1.Amon.hurs.gn', 'CMIP6.ScenarioMIP.CCCma.CanESM5.ssp126.r1i1p1f1.Amon.hurs.gn']

In [4]:
# NBVAL_IGNORE_OUTPUT

# Here the result is keyed by experiment_id.
dsd["ssp370"]["hurs"]

NameError: name 'dsd' is not defined

By default, the ``to_dataset_dict`` method downloads files locally. If you already hold datasets locally, you can specify the ``esg_dataroot`` in the configuration. You can also specify the local_cache where missing datasets will be downloaded.

Please check the [documentation](https://intake-esgf.readthedocs.io/) for more details on how to use `to_dataset_dict`.